# 03 — Local CSV-Based RFECV Feature Selection

This notebook runs **Recursive Feature Elimination with Cross-Validation (RFECV)** outside Google Earth Engine using the sample-features `.csv` exported from Notebook 01 (or any external GEE/Python workflow).

It requires only `lc-framework[ml]` (no Google Earth Engine login needed) and emits a validated **`rfecv_selected_features.json`** artifact consumed directly by both GEE Random Forest and Local ML classifiers ([Notebook 04](04_pixel_classification_gee_and_local.ipynb)).

In [ ]:
# 1. Colab Bootstrap — install [ml] extra only (no GEE required)
# !pip install -q "lc-framework[ml] @ git+https://github.com/<your-org>/LC_Framework.git"

from pathlib import Path

from lc_framework.config import ProjectConfig
from lc_framework.features import load_selected_features, run_rfecv_from_csv

## 2. Point to Your Exported Sample-Features CSV
You can optionally load a `ProjectConfig` to validate class IDs against your `ClassSchema`, or run completely standalone with just `SAMPLE_CSV`.

In [ ]:
# Uncomment in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

CONFIG_PATH = Path("../configs/example_project.yaml")
cfg = ProjectConfig.from_yaml(CONFIG_PATH) if CONFIG_PATH.exists() else None

SAMPLE_CSV = Path("data/samples_features_2025.csv")
OUTPUT_DIR = Path("outputs/rfecv_2025")
CLASS_COL = "class_id"
GROUP_COL = None  # Set to e.g. 'tile_id' or 'plot_id' for spatial group CV

## 3. Run Reproducible RFECV & Save Optimal Feature Subset

In [ ]:
if SAMPLE_CSV.exists():
    result = run_rfecv_from_csv(
        SAMPLE_CSV,
        class_col=CLASS_COL,
        group_col=GROUP_COL,
        config=cfg,
        cv=5,
        scoring="balanced_accuracy",
        min_features_to_select=5,
        n_estimators=200,
        random_state=42,
        output_dir=OUTPUT_DIR,
        artifact_format="json",
    )
    print(f"Samples used: {result.artifact.n_samples} (dropped {result.dropped_rows} invalid rows)")
    print(f"Best CV ({result.artifact.scoring}): {result.artifact.best_score:.4f}")
    print(f"Selected {len(result.selected_features)} / {len(result.artifact.all_candidate_features)} features:")
    print(result.selected_features)
    display(result.feature_rankings.head(15))
else:
    print("Export sample features CSV from Notebook 01 into data/ and re-run this cell.")

## 4. Plot CV Score vs. Number of Features & Reload Artifact

In [ ]:
if SAMPLE_CSV.exists():
    try:
        import matplotlib.pyplot as plt

        scores = result.cv_scores
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(scores["n_features"], scores["mean_test_score"], marker="o")
        ax.fill_between(
            scores["n_features"],
            scores["mean_test_score"] - scores["std_test_score"],
            scores["mean_test_score"] + scores["std_test_score"],
            alpha=0.2,
        )
        ax.axvline(len(result.selected_features), color="crimson", linestyle="--", label="Optimal")
        ax.set_xlabel("Number of Features")
        ax.set_ylabel(result.artifact.scoring)
        ax.set_title("RFECV Cross-Validation Curve")
        ax.legend()
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("Install matplotlib (`lc-framework[viz]`) to display the CV curve plot.")

    # Downstream GEE RF or Local ML loads the exact same artifact:
    artifact = load_selected_features(OUTPUT_DIR / "rfecv_selected_features.json")
    print("Verified reloadable feature list:", list(artifact.selected_features))